In [9]:
import pandas as pd
import numpy as np
import joblib
import pandas as pd

from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor


# Application of Best Model to new Data

## Functions to load new data - keep up to date with the ones used in testing notebook

the feature function must be up to date!!

In [ ]:
def get_targets(bikeData):
    bikeData = bikeData.copy()
    bikeData["target_1h"] = bikeData["BikeCount"].shift(-1)
    bikeData["target_24h"] = bikeData["BikeCount"].shift(-24)

    # remove rows where future target is unknown
    bikeData_model = bikeData.dropna().copy()
    return bikeData_model


def prepare_data(bikeData, target_col):
    # =====================================================
    # 3. Define X and y
    # =====================================================

    X = bikeData.drop(columns=[
        "target_1h",
        "target_24h"
    ])

    y = bikeData[target_col]

    # =====================================================
    # 4. One-hot encode categorical features
    # =====================================================

    categorical_cols = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    print("Before get_dummies:", X.shape)

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

    print("After get_dummies:", X.shape)
    return X, y


def create_features(bikeData):

    bikeData = bikeData.copy()

    # =====================================================
    # 1. Sort chronologically
    # =====================================================

    bikeData = bikeData.sort_values(
        ["Month", "Day", "Hour"]
    ).reset_index(drop=True)

    # =====================================================
    # 2. Lag features
    # =====================================================

    bikeData["lag_1h"] = bikeData["BikeCount"].shift(1)
    bikeData["lag_2h"] = bikeData["BikeCount"].shift(2)
    bikeData["lag_3h"] = bikeData["BikeCount"].shift(3)

    bikeData["lag_24h"] = bikeData["BikeCount"].shift(24)
    bikeData["lag_48h"] = bikeData["BikeCount"].shift(48)
    bikeData["lag_168h"] = bikeData["BikeCount"].shift(168)

    # =====================================================
    # 3. Rolling mean features
    # shift(1) prevents data leakage
    # =====================================================

    bikeData["rolling_mean_3h"] = (
        bikeData["BikeCount"].shift(1).rolling(3).mean()
    )

    bikeData["rolling_mean_6h"] = (
        bikeData["BikeCount"].shift(1).rolling(6).mean()
    )

    bikeData["rolling_mean_12h"] = (
        bikeData["BikeCount"].shift(1).rolling(12).mean()
    )

    bikeData["rolling_mean_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).mean()
    )

    bikeData["rolling_mean_168h"] = (
        bikeData["BikeCount"].shift(1).rolling(168).mean()
    )

    # =====================================================
    # 4. Rolling min/max/std features
    # =====================================================

    bikeData["rolling_std_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).std()
    )

    bikeData["rolling_min_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).min()
    )

    bikeData["rolling_max_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).max()
    )

    # =====================================================
    # 5. Difference / trend features
    # =====================================================

    bikeData["diff_1h"] = (
        bikeData["BikeCount"].shift(1) - bikeData["BikeCount"].shift(2)
    )

    bikeData["diff_24h"] = (
        bikeData["BikeCount"].shift(1) - bikeData["BikeCount"].shift(25)
    )


    # =====================================================
    # 7. Drop missing values
    # =====================================================

    bikeData_model = bikeData.dropna().copy()

    print("Original shape:", bikeData.shape)
    print("Model shape:   ", bikeData_model.shape)

    return bikeData_model



def get_data_for_testing(data):
    data = data.copy()
    data = create_features(data)
    data = get_targets(data)
    X, y = prepare_data(data, "target_1h")
    return X, y

## Retrain the best model on entire train data

In [18]:
# Datei laden
bikeData = pd.read_excel("data/challenge_public_dataset.xlsx")

data_for_1h = bikeData.copy()
data_for_1h = create_features(data_for_1h)
data_for_1h = get_targets(data_for_1h)
X, y = prepare_data(data_for_1h, "target_1h")



# =====================================================
# Retrain model on ALL data
# using known best parameters
# =====================================================

model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    alpha=0.0001,
    learning_rate_init=0.005,
    random_state=43,
    max_iter=10000,
    verbose=True

)
model.fit(X, y)


print("Model retrained on full dataset.")

# =====================================================
# Save model
# =====================================================

joblib.dump(model, "best_model_1h.pkl")
    
print("Model saved.")

##### 24h horizon ########

data_for_24h = bikeData.copy()
data_for_24h = create_features(data_for_24h)
data_for_24h = get_targets(data_for_24h)
X, y = prepare_data(data_for_24h, "target_24h")


## EXCHANGE TO USE THE TRUE BEST MODEL
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    alpha=0.0001,
    learning_rate_init=0.001,
    random_state=43,
    max_iter=10000,
    verbose=True
    

)

model.fit(X, y)

print("Model retrained on full dataset.")

# =====================================================
# Save model
# =====================================================

joblib.dump(model, "best_model_24h.pkl")
    
print("Model saved.")

Original shape: (8760, 26)
Model shape:    (8423, 26)
Before get_dummies: (8399, 26)
After get_dummies: (8399, 57)
Iteration 1, loss = 43215.93228481
Iteration 2, loss = 8647.47080147
Iteration 3, loss = 5563.56332557


/var/folders/2d/rgh939n56knbdhv7tklttl740000gn/T/ipykernel_1134/2986505388.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


Iteration 4, loss = 4412.77599044
Iteration 5, loss = 3999.26208099
Iteration 6, loss = 3628.54662869
Iteration 7, loss = 3314.94030362
Iteration 8, loss = 2994.50350843
Iteration 9, loss = 3023.87224161
Iteration 10, loss = 3097.81707383
Iteration 11, loss = 2892.56958505
Iteration 12, loss = 2752.23814638
Iteration 13, loss = 2583.95448555
Iteration 14, loss = 2676.75557344
Iteration 15, loss = 2794.31335427
Iteration 16, loss = 2704.60544121
Iteration 17, loss = 2623.11627518
Iteration 18, loss = 2499.82519673
Iteration 19, loss = 2359.98599391
Iteration 20, loss = 2667.59023603
Iteration 21, loss = 2461.53642067
Iteration 22, loss = 2370.17409883
Iteration 23, loss = 2313.17848221
Iteration 24, loss = 2262.82864542
Iteration 25, loss = 2414.97158875
Iteration 26, loss = 2345.11188425
Iteration 27, loss = 2230.43564493
Iteration 28, loss = 2268.64734726
Iteration 29, loss = 2186.34919135
Iteration 30, loss = 2146.54887245
Iteration 31, loss = 2149.68490723
Iteration 32, loss = 2091.

/var/folders/2d/rgh939n56knbdhv7tklttl740000gn/T/ipykernel_1134/2986505388.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


Iteration 6, loss = 17297.60388593
Iteration 7, loss = 16440.25844616
Iteration 8, loss = 15799.58070634
Iteration 9, loss = 15260.20182157
Iteration 10, loss = 14809.11284835
Iteration 11, loss = 14476.25911575
Iteration 12, loss = 14166.29563583
Iteration 13, loss = 13854.77004471
Iteration 14, loss = 13693.40733602
Iteration 15, loss = 13437.88976206
Iteration 16, loss = 13142.30310387
Iteration 17, loss = 13035.58426733
Iteration 18, loss = 12942.74854674
Iteration 19, loss = 12760.24910041
Iteration 20, loss = 12543.26196422
Iteration 21, loss = 12507.34451811
Iteration 22, loss = 12199.70668728
Iteration 23, loss = 12123.72379790
Iteration 24, loss = 11969.92163801
Iteration 25, loss = 12070.03134035
Iteration 26, loss = 11781.42725799
Iteration 27, loss = 11597.25579123
Iteration 28, loss = 11620.50479295
Iteration 29, loss = 11875.93087405
Iteration 30, loss = 11349.89194357
Iteration 31, loss = 11374.90950736
Iteration 32, loss = 11224.98479560
Iteration 33, loss = 10975.46291

## Test model on new data

In [ ]:
MODEL_PATH = "best_model_1h.pkl"
DATA_PATH = "data/challenge_public_dataset.xlsx"
from sklearn.metrics import mean_squared_error

# Datei laden
bikeDataNew = pd.read_excel(DATA_PATH)


#### Model für 1h Vorhersage #### 
data_for_1h = bikeDataNew.copy()
X_1h, y_1h = get_data_for_testing(data_for_1h)

# =====================================================
# Reload model
# =====================================================

model_1h = joblib.load(MODEL_PATH)

print("Model reloaded.")

# Predict
y_pred_1h = model_1h.predict(X_1h)

# Evaluate
mse_1h = mean_squared_error(y_1h, y_pred_1h)

print(f"1h Forecast MSE: {mse_1h:.4f}")


#### Model für 24h Vorhersage #### 

#Load model
#data_for_24h = bikeDataNew.copy()
#X_24h, y_24h = get_data_for_testing(data_for_24h)

# Load trained model
#model_24h = joblib.load("best_model_24h.pkl")

# Predict
#y_pred_24h = model_24h.predict(X_24h)

# Evaluate
#mse_24h = mean_squared_error(y_24h, y_pred_24h)

#print(f"24h Forecast MSE: {mse_24h:.4f}")

NameError: name 'pd' is not defined